# The 3-Qubit Repetition Code

This notebook reviews the 3-qubit repetition code, one of the most basic quantum error correction algorithms. We will see how the syndrome identifies a single bit-flip error and why the code fails against phase flips. Logical failure will depend on the physical bit-flip rate. The tutorial builds the code for understanding these ideas, from deriving a syndrome table to simulating noisy runs and measuring decoding success rates versus physical error rates.

In [1]:
# Import Dependencies 
import numpy as np
import matplotlib.pyplot as plt
from itertools import product as iproduct

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit.quantum_info import Statevector

sim = AerSimulator()

### 1. Logical encoding

We define the \textbf{logical} basis states as:

$$\ket{0_L} = \ket{000}, \quad \ket{1_L} = \ket{111}$$

An arbitrary logical qubit can be written as:

$$\ket{\psi} = \alpha \ket{0} + \beta \ket{1}$$

and encoded in the logical basis a:

$$\ket{\psi_L} = \alpha \ket{000} + \beta \ket{111} $$

The redundancy in the encoding lets us later detect disagreement between neighboring qubit states without revealing the logical information $\alpha, \beta$.

In [4]:
def encode_logical_bit(bit: int) -> list[int]:
    """Encode a classical logical bit into the 3-bit repetition code."""
    assert bit in (0, 1), "bit must be 0 or 1"
    return [bit, bit, bit]

print("Logical 0 ->", encode_logical_bit(0))
print("Logical 1 ->", encode_logical_bit(1))

Logical 0 -> [0, 0, 0]
Logical 1 -> [1, 1, 1]


### 1.1  Encoding Circuit

The quantum encoding is implemented with two CNOT gates from the data qubit $q_0$ onto the ancilla qubits $q_1$ and $q_2$:

$$|\psi\rangle|00\rangle \xrightarrow{\text{CNOT}_{01}} \xrightarrow{\text{CNOT}_{02}} |\psi_L\rangle$$

The state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ on $q_0$ fans out into the three-qubit codeword.

In [ ]:
def build_encoding_circuit(alpha: complex = 1.0, beta: complex = 0.0) -> QuantumCircuit:
    """
    Build the 3-qubit repetition code encoding circuit.
    alpha|0> + beta|1>  ->  alpha|000> + beta|111>
    """
    norm = np.sqrt(abs(alpha)**2 + abs(beta)**2)
    alpha, beta = alpha / norm, beta / norm

    qc = QuantumCircuit(3, name="encode")
    # Initialise q0 to the desired logical state
    qc.initialize([alpha, beta], 0)
    # Fan-out
    qc.cx(0, 1)
    qc.cx(0, 2)
    return qc

enc = build_encoding_circuit(alpha=1/np.sqrt(2), beta=1/np.sqrt(2))
enc.draw("mpl")

In [ ]:
# To:Do - Verify: for |0_L> the only non-zero amplitude should be |000>


TypeError: type numpy.complex128 doesn't define __round__ method

### Error Model

To simulate random errors that flip a given qubit, we will build a function that applies a random bit flip to the 3-bit classical string or simulated quantum state. For this introduction, the classical bit flip willl be sufficient to highlight the mechanism of the error detection algorithm.

The noise funciton will include the following:
* Each qubit will be independently affected by a random bit flip error, captured by an $X$-gate, which is applied with probability $p$.
* One round of syndrom extraction is used

### Stabilizers and syndrome extraction

Here, let us introduce the followin stabilizers:

$S_1 = Z_1Z_1, \quad S_2 = Z_2Z_3$

The code state are eigenstates of both stabilizers with eigenvalue $+1$.